In [ ]:
import os
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from transformers import BertTokenizerFast, BertForSequenceClassification, AdamW
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import classification_report
import nltk

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Function to read files from a folder
def read_files(folder):
    texts = []
    with open(os.path.join(folder, 'processed.review.balanced'), 'r', encoding='utf-8') as f:
        texts.extend(f.readlines())
    return texts

# Load data from each folder
categories = ['apparel', 'automotive', 'baby', 'beauty', 'books', 'camera_&_photo', 
              'cell_phones_&_service', 'computer_&_video_games', 'dvd', 'electronics', 
              'gourmet_food', 'grocery', 'health_&_personal_care', 'jewelry_&_watches', 
              'kitchen_&_housewares', 'magazines', 'music', 'musical_instruments', 
              'office_products', 'outdoor_living', 'software', 'sports_&_outdoors', 
              'tools_&_hardware', 'toys_&_games', 'video']

texts = []
labels = []
for category in categories:
    folder_path = os.path.join('/kaggle/input/multidomain-sentiment-analysis-dataset/unprocessed/sorted_data', category)
    category_texts = read_files(folder_path)
    texts.extend(category_texts)
    labels.extend([categories.index(category)] * len(category_texts))

# NLTK stopwords
nltk.download('stopwords')
stopwords = set(nltk.corpus.stopwords.words('english'))

# Remove stopwords
filtered_texts = []
for text in texts:
    words = text.split()
    filtered_text = ' '.join([word for word in words if word.lower() not in stopwords])
    filtered_texts.append(filtered_text)

# Tokenizer initialization
tokenizer = BertTokenizerFast.from_pretrained('bert-base-uncased')

# Tokenize texts
tokenized_texts = tokenizer(filtered_texts, padding=True, truncation=True, return_tensors="pt")

# Convert tokenized texts to PyTorch tensors
input_ids = tokenized_texts['input_ids']
attention_masks = tokenized_texts['attention_mask']
labels = torch.tensor(labels)

# Split data into training and testing sets
input_ids_train, input_ids_test, attention_masks_train, attention_masks_test, labels_train, labels_test = train_test_split(
    input_ids, attention_masks, labels, test_size=0.2, random_state=42)

# Create TensorDatasets
train_dataset = TensorDataset(input_ids_train, attention_masks_train, labels_train)
test_dataset = TensorDataset(input_ids_test, attention_masks_test, labels_test)

# DataLoader for training and testing sets
batch_size = 16
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# Load pre-trained BERT model
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=len(categories)).to(device)

# Optimizer and learning rate scheduler
optimizer = AdamW(model.parameters(), lr=2e-5)
epochs = 3
total_steps = len(train_loader) * epochs

# Fine-tuning BERT
for epoch in range(epochs):
    total_loss = 0
    total_correct = 0
    total_samples = 0
    model.train()
    for batch in train_loader:
        input_ids, attention_masks, labels = batch
        input_ids, attention_masks, labels = input_ids.to(device), attention_masks.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask=attention_masks, labels=labels)
        loss = outputs.loss
        logits = outputs.logits
        total_loss += loss.item()
        _, predicted = torch.max(logits, 1)
        total_correct += (predicted == labels).sum().item()
        total_samples += labels.size(0)
        loss.backward()
        optimizer.step()

    avg_loss = total_loss / len(train_loader)
    accuracy = total_correct / total_samples
    print(f'Epoch {epoch + 1}/{epochs}, Train Loss: {avg_loss:.4f}, Train Accuracy: {accuracy:.4f}')

    # Evaluation on test set
    model.eval()
    with torch.no_grad():
        total_correct_test = 0
        total_samples_test = 0
        for batch in test_loader:
            input_ids, attention_masks, labels = batch
            input_ids, attention_masks, labels = input_ids.to(device), attention_masks.to(device), labels.to(device)
            outputs = model(input_ids, attention_mask=attention_masks)
            _, predicted = torch.max(outputs.logits, 1)
            total_correct_test += (predicted == labels).sum().item()
            total_samples_test += labels.size(0)
    accuracy_test = total_correct_test / total_samples_test
    print(f'Test Accuracy: {accuracy_test:.4f}')

# Evaluation
model.eval()
preds = []
true_labels = []
for batch in test_loader:
    input_ids, attention_masks, labels = batch
    input_ids, attention_masks, labels = input_ids.to(device), attention_masks.to(device), labels.to(device)
    with torch.no_grad():
        outputs = model(input_ids, attention_mask=attention_masks)
    logits = outputs.logits
    preds.extend(logits.argmax(dim=1).cpu().numpy())
    true_labels.extend(labels.cpu().numpy())

# Classification report
print(classification_report(true_labels, preds))